# Challenge 06 - CNN

Identification of the specific animal (12 classes, individual dogs and cats) depicted in each image, with a **MobileNet V1 "small"** (input 128x128, width multiplier 0.5) trained with TensorFlow / Keras. The base network is fixed by the challenge rules; only the final layers, the training procedure and the data augmentation can be changed.

## Setup

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
from joblib import Memory
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import itertools
import matplotlib.patches as mpatches
import skimage
from skimage import transform
from packaging import version

# Configuration
db_path = '../../DBs/CaniGatti_ML18_Es8_2020'
train_filelist = 'MulticlassTrainingSet.txt'
test_filelist = 'MulticlassTestSet.txt'
label_filelist = db_path + '/MulticlassLabels.txt'
result_path = '../Es8Predictions.txt'
checkpoints_path = './saved_models'
exp_path = './exp_cache'

class_num = 12
validation_size = 200

# MobileNet V1 'small': fixed by the challenge rules, do not change
image_side = 128
width_multiplier = 0.5
IMG_SHAPE = (image_side, image_side, 3)

# Model (to tune)
pretrained = True        # True: ImageNet weights, False: random initialization
head_dropout = 0.2       # dropout before the final Dense layer

# Training (to tune)
minibatch_size = 50
n_epochs = 20
learning_rate = 0.0025
warmup_epochs = 0        # > 0: train only the new head first, with the base network frozen
warmup_lr = 0.01
use_augmentation = True

seed = 1234
tf.keras.utils.set_random_seed(seed)

# Disk cache that speeds up the repeated loading / resizing of the images
memory = Memory(exp_path, verbose=0)
Path(checkpoints_path).mkdir(exist_ok=True, parents=True)
best_model_path = str(Path(checkpoints_path) / 'challenge06_best.keras')

## Utility Functions

In [ ]:
def internal_load_labeled_dataset(filelist_name, db_folder_path):
    files = []
    labels = []

    db_folder_path = Path(db_folder_path)

    with open(str(db_folder_path / filelist_name), 'r') as f:
        for i, line in enumerate(f):
            path, label = line.split()
            files.append(db_folder_path / path)
            labels.append(int(label))

    images = [plt.imread(file_path) for file_path in files]

    return images, np.array(labels)


def load_labeled_dataset(filelist_name, db_folder_path, cache=None):
    if cache is not None:
        cached_function = cache.cache(internal_load_labeled_dataset)
    else:
        cached_function = internal_load_labeled_dataset

    return cached_function(filelist_name, db_folder_path)


def internal_load_unlabeled_dataset(filelist_name, db_folder_path):
    files = []

    db_folder_path = Path(db_folder_path)

    with open(str(db_folder_path / filelist_name), 'r') as f:
        for i, line in enumerate(f):
            path = line.strip()
            files.append(db_folder_path / path)

    images = [plt.imread(file_path) for file_path in files]

    return images


def load_unlabeled_dataset(filelist_name, db_folder_path, cache=None):
    if cache is not None:
        cached_function = cache.cache(internal_load_unlabeled_dataset)
    else:
        cached_function = internal_load_unlabeled_dataset

    return cached_function(filelist_name, db_folder_path)


def internal_resize_images(imgs, size_h, size_w):
    # The input images are assumed to be RGB (3 channels) with shape (h, w, c)
    num_imgs = len(imgs)
    resized_images = np.empty((num_imgs, size_h, size_w, 3), dtype=np.float32)

    for i in range(num_imgs):
        if version.parse(skimage.__version__) < version.parse("0.14.0"):
            res_img = transform.resize(imgs[i], (size_h, size_w), mode='reflect')
        else:
            res_img = transform.resize(imgs[i], (size_h, size_w), mode='reflect', anti_aliasing=True)
        resized_images[i] = res_img

    return resized_images


def resize_images(imgs, size_h, size_w, cache=None):
    if cache is not None:
        cached_function = cache.cache(internal_resize_images)
    else:
        cached_function = internal_resize_images

    return cached_function(imgs, size_h, size_w)


def legend_drawing(numFigure, colors, labels, loc):
    plt.figure(numFigure)
    items = [mpatches.Rectangle((0, 0), 1, 1, fc=c) for c in colors]
    plt.legend(items, labels, loc=loc)
    plt.draw()


def plot_performance_curves(epochs_training_loss, epochs_training_accuracy, epochs_validation_accuracy):
    _, ax1 = plt.subplots(dpi=96, figsize=(8, 6))
    ax2 = ax1.twinx()
    ax1.set_ylim(0, epochs_training_loss[0] * 1.1)
    ax2.set_ylim(40, 100)
    ax1.plot(range(0, len(epochs_training_loss)), epochs_training_loss, 'r')
    ax2.plot(range(0, len(epochs_training_accuracy)), epochs_training_accuracy, 'b')
    ax2.plot(range(0, len(epochs_validation_accuracy)), epochs_validation_accuracy, 'g')
    legend_drawing(1, ('r', 'b', 'g', 'c'), ("Loss", "Training Accuracy", "Validation Accuracy"), 0)
    ax1.set_xlabel('# Epoche')
    ax1.set_ylabel('Loss')
    ax2.set_ylabel('Accuratezza %')
    plt.show()


def plot_confusion_matrix(y_true, y_pred, classes, title='Confusion matrix', cmap=plt.cm.Blues, figsize=(8, 6)):
    cm = confusion_matrix(y_true, y_pred)
    cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    plt.figure(dpi=96, figsize=figsize)
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()

## Helper Functions

Model definition, data augmentation, dataset creation, training and export.

- **Model**: MobileNet V1 (`include_top=False`, average pooling) followed by a `Dropout` and a `Dense` softmax layer with `class_num` units. The base network is not modified.
- **Augmentation**: random transformations applied on the training images only, never on the validation or test images.
- **Training**: optional warm-up of the head with the base network frozen, then training of the whole network.

In [ ]:
# Augmentation applied on the training images only
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomTranslation(0.05, 0.05),
])


def build_model():
    base_model = tf.keras.applications.MobileNet(input_shape=IMG_SHAPE,
                                                 alpha=width_multiplier,
                                                 include_top=False,
                                                 pooling='avg',
                                                 weights='imagenet' if pretrained else None)
    base_model.trainable = True
    inputs = tf.keras.Input(shape=IMG_SHAPE)
    x = base_model(inputs)
    x = tf.keras.layers.Dropout(head_dropout)(x)
    outputs = tf.keras.layers.Dense(class_num, activation='softmax')(x)
    return tf.keras.Model(inputs, outputs), base_model


def compile_model(model, lr):
    model.compile(loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                  optimizer=tf.keras.optimizers.SGD(learning_rate=lr),
                  metrics=['accuracy'])


def make_dataset(x, y, training):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(len(x), seed=seed).batch(minibatch_size)
        if use_augmentation:
            ds = ds.map(lambda i, l: (augmentation(i, training=True), l), num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = ds.batch(minibatch_size)
    return ds.prefetch(tf.data.AUTOTUNE)


def fit_model(model, base_model, train_ds, epochs, val_ds=None, callbacks=None):
    if warmup_epochs > 0:
        base_model.trainable = False
        compile_model(model, warmup_lr)
        model.fit(train_ds, epochs=warmup_epochs, verbose=0)
        base_model.trainable = True
    compile_model(model, learning_rate)
    return model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=callbacks, verbose=2)


def save_predictions(predictions, path):
    np.savetxt(path, np.asarray(predictions, dtype=int), fmt='%i')
    print(f'Predictions successfully saved to {path}')

## Data Loading

The training set is loaded from `MulticlassTrainingSet.txt` and resized to 128x128. A stratified validation set is held out from it to tune the hyperparameters. The test set (`MulticlassTestSet.txt`, no labels) is loaded only at the end, in the export section. No external images are used.

In [ ]:
print('Loading ...')
start = time.time()
raw_images_x, label_y = load_labeled_dataset(train_filelist, db_path, cache=memory)
label_names = [line.rstrip('\n') for line in open(label_filelist, 'r')]
resized_image_x = resize_images(raw_images_x, image_side, image_side, cache=memory)
print('Loaded and resized %d images in %.2f s.' % (len(resized_image_x), time.time() - start))
print('Image tensor shape:', np.shape(resized_image_x), '- value range: %.2f, %.2f' % (np.min(resized_image_x), np.max(resized_image_x)))
for i, name in enumerate(label_names):
    print('{}: {}'.format(name, np.count_nonzero(label_y == i)))

train_x, validation_x, train_y, validation_y = train_test_split(resized_image_x, label_y,
                                                                test_size=validation_size,
                                                                stratify=label_y,
                                                                random_state=seed)
print('Training patterns: %d, validation patterns: %d' % (len(train_x), len(validation_x)))

## Model: MobileNet V1 small

MobileNet V1 with resolution 128 and width multiplier 0.5, followed by a `Dropout` and a `Dense` softmax layer with 12 units. TODO: describe the final choices (pre-trained weights, head, trainable layers).

In [ ]:
model, base_model = build_model()
model.summary()

## Model Evaluation

The network is trained on the training split and evaluated on the validation split after every epoch. The model of the epoch with the best validation accuracy is saved, so the saved model is the best one even if `n_epochs` is larger than the optimal value. Note that the choices made on this validation set make its accuracy slightly optimistic.

In [ ]:
train_ds = make_dataset(train_x, train_y, training=True)
valid_ds = make_dataset(validation_x, validation_y, training=False)

checkpoint = tf.keras.callbacks.ModelCheckpoint(best_model_path, monitor='val_accuracy',
                                                save_best_only=True, mode='max')

model, base_model = build_model()
t_start = time.time()
history = fit_model(model, base_model, train_ds, n_epochs, val_ds=valid_ds, callbacks=[checkpoint])
print('Training time: %.2f s' % (time.time() - t_start))

h = history.history
best_epoch = int(np.argmax(h['val_accuracy'])) + 1
print('Best validation accuracy: %.2f %% at epoch %d' % (100 * max(h['val_accuracy']), best_epoch))
plot_performance_curves(h['loss'],
                                         [a * 100 for a in h['accuracy']],
                                         [a * 100 for a in h['val_accuracy']])

In [ ]:
# Confusion matrix of the best saved model on the validation set
best_model = tf.keras.models.load_model(best_model_path)
valid_probs = best_model.predict(valid_ds, verbose=0)
predicted_y = valid_probs.argmax(axis=1)
print('Validation accuracy: %.2f %%' % (100 * accuracy_score(validation_y, predicted_y)))
plot_confusion_matrix(validation_y, predicted_y, label_names, figsize=(10, 8))

## Hyperparameter Tuning

## Error Analysis

In [ ]:
errors = np.where(predicted_y != validation_y)[0]
print('Misclassified validation images: %d / %d' % (len(errors), len(validation_y)))

## Final Configuration

## Prediction and Export

The network is retrained on all the training images with the configuration chosen above (`n_epochs` set to the best epoch found on the validation set, or kept as it is if the best model was the last one), and the same preprocessing (resizing to 128x128) is applied to the test set. Each test image is classified independently of the others. The predictions are saved in `Es8Predictions.txt`, one class index per line.

In [ ]:
final_epochs = n_epochs

# Test set (no labels): same preprocessing as the training set
raw_test_images_x = load_unlabeled_dataset(test_filelist, db_path, cache=memory)
resized_test_image_x = resize_images(raw_test_images_x, image_side, image_side, cache=memory)
test_ds = tf.data.Dataset.from_tensor_slices((resized_test_image_x,)).batch(minibatch_size)

# Retraining on all the training images
tf.keras.utils.set_random_seed(seed)
full_ds = make_dataset(np.asarray(resized_image_x), label_y, training=True)
final_model, final_base = build_model()
fit_model(final_model, final_base, full_ds, final_epochs)

test_probs = final_model.predict(test_ds, verbose=0)
predictions = test_probs.argmax(axis=1)

assert len(predictions) == len(resized_test_image_x)
print('Predicted images per class:', np.bincount(predictions, minlength=class_num))
save_predictions(predictions, result_path)